<a href="https://colab.research.google.com/github/dsci3d/messyverse/blob/main/notebooks/03_pdf-extraktion.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="In Colab oeffnen"/></a>

# Rechnungen automatisch auslesen -- aus PDF

Im Ordner `erwerbung/` liegen Lieferantenrechnungen als PDF. Dein Auftrag: aus jeder Rechnung die Rechnungs-Nummer, den Lieferanten und den Rechnungsbetrag auslesen und sammeln. Achtung: eine Rechnung enthaelt bewusst eine Position **ohne** Betrag -- ein typischer Sonderfall.

In [ ]:
# SETUP (Black Box -- einmal ausfuehren). Holt deine Arbeitskopie des Uebungsuniversums.
import os, shutil, subprocess
ZIEL = "/content/messyverse"
if os.path.exists(ZIEL):
    shutil.rmtree(ZIEL)              # idempotent: alte Kopie weg, dann frisch klonen
subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                "https://github.com/dsci3d/messyverse.git", ZIEL], check=True)
os.chdir(ZIEL)
anzahl = sum(len(d) for r, _, d in os.walk(ZIEL) if ".git" not in r)
print(f"Arbeitskopie: {anzahl} Dateien unter {ZIEL}")

## Werkzeug nachladen

Zum Lesen von PDF-Text braucht Colab eine kleine Bibliothek.

In [ ]:
!pip install -q pypdf

## Schritt 1 -- die Lage ansehen

Sieh dir den Text einer Rechnung an, damit du weisst, woran die KI die Felder erkennt.

In [ ]:
import os
from pypdf import PdfReader
print(sorted(os.listdir("erwerbung")))
print(PdfReader("erwerbung/RE-2025-0003.pdf").pages[0].extract_text())

## Schritt 2 -- die KI prompten

> *Lies jede PDF in `erwerbung/` mit pypdf. Zieh aus dem Text die Rechnungs-Nummer (Zeile 'RECHNUNG ...'), den Lieferanten und den Rechnungsbetrag (Zeile 'Rechnungsbetrag: ... EUR', als Zahl). Baue ein dict `ergebnis`: Rechnungs-Nummer -> {'betrag': float, 'lieferant': str}. Behandle den Fall, dass eine Position '(Betrag fehlt)' enthaelt.*

In [ ]:
# Code deines KI-Assistenten hier einfuegen.
# Ziel: dict `ergebnis` -> 'RE-2025-XXXX': {'betrag': float, 'lieferant': str}


## Schritt 3 -- gegen das Universum pruefen (Black Box)

In [ ]:
import json
soll = json.load(open("loesungen/pdf_extraktion.golden.json"))
try:
    ergebnis
except NameError:
    print("Noch kein `ergebnis`.")
else:
    fehlen = [r for r in soll if r not in ergebnis]
    betrag_ab = [r for r in soll if r in ergebnis and round(float(ergebnis[r].get("betrag", -1)), 2) != soll[r]["betrag_eur"]]
    if not fehlen and not betrag_ab:
        print(f"OK -- alle {len(soll)} Rechnungsbetraege stimmen.")
    else:
        if fehlen: print("Diese Rechnungen fehlen:", fehlen)
        if betrag_ab: print("Bei diesen Rechnungen weicht der Betrag ab:", betrag_ab)

## Wenn es klemmt -- die Fehlerschleife

Lief der Code auf einen Fehler, oder meldet die Pruefzelle Abweichungen? Dann **nicht selbst reparieren**. Kopiere die Fehlermeldung oder die Abweichungs-Liste, gib sie deinem Assistenten zurueck und bitte um eine korrigierte Fassung. Die haeufigsten Ursachen sind uebersehene Sonderfaelle -- leere Eintraege, fehlende Felder, ungewohnte Schreibweisen.

## Weiterdenken

Die Rechnungen tragen das Datum in **gemischten Formaten** (15.03.2025, 2025-04-08, 3. Mai 2025). Lass deine KI zusaetzlich das Datum auslesen und einheitlich als JJJJ-MM-TT normieren -- die gleiche Aufgabe begegnet dir beim Datei-Sortieren wieder.